# submission


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ml_final_project


In [ ]:
import os
if not os.path.exists('/content/ml_final_project'):
    !git clone -q https://github.com/ochiga07/ml_final_project.git /content/ml_final_project
import sys
sys.path.append('/content/ml_final_project')

from colab_setup import setup_project
drive_repo = setup_project(repo_url="https://github.com/ochiga07/ml_final_project.git")


In [ ]:
!pip install -q mlflow dagshub


In [ ]:
import pandas as pd
import mlflow
import dagshub
import warnings
warnings.filterwarnings('ignore')

if 'DAGSHUB_USER_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        os.environ['DAGSHUB_USER_TOKEN'] = userdata.get('DAGSHUB_USER_TOKEN')
    except Exception:
        pass

dagshub.init(repo_owner='aochi23', repo_name='ml_final_project', mlflow=True)


In [ ]:
# get best run from mlflow
client = mlflow.tracking.MlflowClient()
exp = client.search_experiments()
ids = [e.experiment_id for e in exp]

runs = mlflow.search_runs(
    experiment_ids=ids,
    order_by=["metrics.val_wmae ASC"],
    max_results=1
)

best = runs.iloc[0]
print(f"best run: {best['tags.mlflow.runName']}")
print(f"wmae: {best['metrics.val_wmae']:.2f}")


In [ ]:
# load model based on run name
name = best['tags.mlflow.runName']
reg_name = "Walmart_" + name.split('_')[0]
uri = f"models:/{reg_name}/latest"

model = mlflow.sklearn.load_model(uri)
print("loaded")


In [ ]:
test_df = pd.read_csv(f'{drive_repo}/data/test.csv.zip', parse_dates=['Date'])
preds = model.predict(test_df)
print("predicted")


In [ ]:
# kaggle id format is store_dept_date
sub = pd.DataFrame({
    'Id': test_df['Store'].astype(str) + '_' + test_df['Dept'].astype(str) + '_' + test_df['Date'].dt.strftime('%Y-%m-%d'),
    'Weekly_Sales': preds
})

sub.to_csv(f'{drive_repo}/submission.csv', index=False)
sub.head()
